# NetworkX

In [2]:
import numpy as np
import scipy.sparse as sps
import gzip
import uuid

from sklearn.datasets import load_svmlight_file
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.neighbors import NearestNeighbors

import dimod
from dwave.samplers import SimulatedAnnealingSampler


# =========================================================
# CONFIG
# =========================================================

BASE_PATH = "/config/workspace/datasets/Tasks/2A/noise_injected_dataset/embeddings"
FOLD = 2
KEEP_RATIO = 0.3


# =========================================================
# LOAD DATA
# =========================================================

def load_gz(path):
    X, y = load_svmlight_file(path)
    return X.toarray(), y.astype(int)


path = f"{BASE_PATH}/train{FOLD}.gz"
X, y = load_gz(path)


# =========================================================
# TRAIN / VAL SPLIT
# =========================================================

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

n_train = len(y_train)


# =========================================================
# STEP 1 — LOSS MODEL
# =========================================================

clf = LogisticRegression(max_iter=500)
clf.fit(X_train, y_train)

probs = clf.predict_proba(X_train)

loss = -np.log(
    probs[np.arange(n_train), y_train] + 1e-9
)


# =========================================================
# STEP 2 — GMM CLEAN PROB
# =========================================================

gmm = GaussianMixture(n_components=2, random_state=42)
gmm.fit(loss.reshape(-1, 1))

posterior = gmm.predict_proba(loss.reshape(-1, 1))

clean_prob = posterior[:, np.argmin(gmm.means_.ravel())]
clean_prob = clean_prob / (clean_prob.max() + 1e-9)


# =========================================================
# STEP 3 — kNN FEATURES
# =========================================================

k = 10

knn = NearestNeighbors(n_neighbors=k, metric="cosine")
knn.fit(X_train)

distances, indices = knn.kneighbors(X_train)


density = 1 / (np.mean(distances[:, 1:], axis=1) + 1e-9)
density = density / density.max()

centrality = np.sum(1 - distances, axis=1)
centrality = centrality / centrality.max()


# =========================================================
# STEP 4 — CLASS BALANCE
# =========================================================

class_count = np.bincount(y_train)
class_weight = 1.0 / (class_count + 1e-9)

cw = np.array([class_weight[c] for c in y_train])
cw = cw / cw.max()


# =========================================================
# STEP 5 — REDUNDANCY
# =========================================================

redundancy = np.zeros(n_train)

for i in range(n_train):
    redundancy[i] = len(set(indices[i]))

redundancy = redundancy / redundancy.max()


# =========================================================
# STEP 6 — QUBO SCORE
# =========================================================

alpha = 2.0
beta = 1.2
gamma = 1.0
delta = 1.5
epsilon = 1.0

score = (
    alpha * clean_prob +
    beta * density +
    gamma * centrality +
    epsilon * cw -
    delta * redundancy
)

Q = {(i, i): -score[i] for i in range(n_train)}


# =========================================================
# STEP 7 — BQM
# =========================================================

bqm = dimod.BinaryQuadraticModel.from_qubo(Q)

k_keep = int(KEEP_RATIO * n_train)

bqm.update(
    dimod.generators.combinations(
        n_train,
        k_keep,
        strength=bqm.maximum_energy_delta()
    )
)


# =========================================================
# QA / SA SUBMISSION (QCLEF TRACKING)
# =========================================================

def solve_qa(bqm):
    from qclef import qa_access as qa
    from dwave.samplers import SimulatedAnnealingSampler

    sampler = SimulatedAnnealingSampler()

    sampleset = qa.submit(
        sampler,
        SimulatedAnnealingSampler.sample,
        bqm,
        label="2A-Instance_Selection",
        num_reads=5
    )

    best = sampleset.first.sample
    problem_id = sampleset.info.get("problem_id", str(uuid.uuid4()))

    return best, problem_id


# =========================================================
# STEP 8 — SOLVE + SUBMISSION)
# =========================================================

# QClef submission run (for competition tracking)
best, problem_id = solve_qa(bqm)


# =========================================================
# STEP 9 — SELECT INSTANCES
# =========================================================

selected = [i for i, v in best.items() if v == 1]

print("Selected samples:", len(selected))
print("Problem ID:", problem_id)


# =========================================================
# STEP 10 — FINAL MODEL
# =========================================================

clf_final = LogisticRegression(max_iter=500)
clf_final.fit(X_train[selected], y_train[selected])

pred = clf_final.predict(X_val)

f1 = f1_score(y_val, pred, average="macro")

print("FOLD:", FOLD)
print("MACRO-F1:", f1)

Selected samples: 949
Problem ID: SA-7061
FOLD: 2
MACRO-F1: 0.9515256159158849


# On All Folds

In [1]:
import numpy as np
import scipy.sparse as sps
import uuid
import os

from sklearn.datasets import load_svmlight_file
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import f1_score

import dimod
from dwave.samplers import SimulatedAnnealingSampler


# =========================================================
# CONFIG
# =========================================================

BASE_PATH = "/config/workspace/datasets/Tasks/2A/noise_injected_dataset/embeddings"

CFG = {
    "group_name": "NUCES-FAST",
    "submission_id": "DivideMix",
    "submission_dir": "/config/workspace/submissions"
}

os.makedirs(CFG["submission_dir"], exist_ok=True)

KEEP_RATIO = 0.3
K_FOLDS = 5


# =========================================================
# LOAD FUNCTION
# =========================================================

def load_gz(path):
    X, y = load_svmlight_file(path)
    return X.toarray(), y.astype(int)


# =========================================================
# QClef SOLVER (IMPORTANT FIX)
# =========================================================

def solve_sa_qclef(bqm):

    from qclef import qa_access as qa

    sampler = SimulatedAnnealingSampler()

    sampleset = qa.submit(
        sampler,
        SimulatedAnnealingSampler.sample,
        bqm,
        label="2A-Instance_Selection",
        num_reads=50
    )

    best = sampleset.first.sample
    problem_id = sampleset.info.get("problem_id", str(uuid.uuid4()))

    return best, problem_id


# =========================================================
# SUBMISSION WRITER
# =========================================================

def write_submission(selected_ids, problem_ids, fold):

    fname = f"VAder_{fold}_SA_{CFG['group_name']}_{CFG['submission_id']}.txt"
    path = os.path.join(CFG["submission_dir"], fname)

    with open(path, "w") as f:

        for idx in selected_ids:
            f.write(f"{idx}\n")

        f.write(str(problem_ids) + "\n")

    print("Saved:", path)


# =========================================================
# PIPELINE PER FOLD
# =========================================================

for FOLD in range(K_FOLDS):

    print("\n========================")
    print("FOLD:", FOLD)
    print("========================")

    # -------------------------
    # LOAD FULL DATA (NO SPLIT)
    # -------------------------
    X, y = load_gz(f"{BASE_PATH}/train{FOLD}.gz")
    n_train = len(y)


    # -------------------------
    # STEP 1: LOSS MODEL
    # -------------------------
    clf = LogisticRegression(max_iter=500)
    clf.fit(X, y)

    probs = clf.predict_proba(X)

    loss = -np.log(probs[np.arange(n_train), y] + 1e-9)


    # -------------------------
    # STEP 2: GMM CLEAN PROB
    # -------------------------
    gmm = GaussianMixture(n_components=2, random_state=42)
    gmm.fit(loss.reshape(-1, 1))

    posterior = gmm.predict_proba(loss.reshape(-1, 1))

    clean_prob = posterior[:, np.argmin(gmm.means_.ravel())]
    clean_prob = clean_prob / (clean_prob.max() + 1e-9)


    # -------------------------
    # STEP 3: kNN STRUCTURE
    # -------------------------
    k = 10
    knn = NearestNeighbors(n_neighbors=k, metric="cosine")
    knn.fit(X)

    distances, indices = knn.kneighbors(X)

    density = 1 / (np.mean(distances[:, 1:], axis=1) + 1e-9)
    density = density / density.max()

    centrality = np.sum(1 - distances, axis=1)
    centrality = centrality / centrality.max()


    # -------------------------
    # STEP 4: CLASS BALANCE
    # -------------------------
    class_count = np.bincount(y)
    class_weight = 1.0 / (class_count + 1e-9)

    cw = np.array([class_weight[c] for c in y])
    cw = cw / cw.max()


    # -------------------------
    # STEP 5: REDUNDANCY
    # -------------------------
    redundancy = np.zeros(n_train)

    for i in range(n_train):
        redundancy[i] = len(set(indices[i]))

    redundancy = redundancy / redundancy.max()


    # -------------------------
    # STEP 6: QUBO SCORE
    # -------------------------
    alpha = 2.0
    beta = 1.2
    gamma = 1.0
    delta = 1.5
    epsilon = 1.0

    score = (
        alpha * clean_prob +
        beta * density +
        gamma * centrality +
        epsilon * cw -
        delta * redundancy
    )

    Q = {(i, i): -score[i] for i in range(n_train)}

    bqm = dimod.BinaryQuadraticModel.from_qubo(Q)

    k_keep = int(KEEP_RATIO * n_train)

    bqm.update(
        dimod.generators.combinations(
            n_train,
            k_keep,
            strength=bqm.maximum_energy_delta()
        )
    )


    # -------------------------
    # STEP 7: QClef SOLVER
    # -------------------------
    best, problem_id = solve_sa_qclef(bqm)

    selected = [i for i, v in best.items() if v == 1]


    # -------------------------
    # STEP 8: QUICK CHECK (optional)
    # -------------------------
    clf_final = LogisticRegression(max_iter=500)
    clf_final.fit(X[selected], y[selected])

    pred = clf_final.predict(X)
    f1 = f1_score(y, pred, average="macro")

    print("Selected samples:", len(selected))
    print("Self F1:", f1)
    print("Problem ID:", problem_id)


    # -------------------------
    # STEP 9: SUBMISSION FILE
    # -------------------------
    write_submission(
        selected_ids=selected,
        problem_ids=[problem_id],
        fold=FOLD
    )


FOLD: 0
Selected samples: 1186
Self F1: 0.9513249579070069
Problem ID: SA-7063
Saved: /config/workspace/submissions/VAder_0_SA_NUCES-FAST_DivideMix.txt

FOLD: 1
Selected samples: 1186
Self F1: 0.9378733658405978
Problem ID: SA-7064
Saved: /config/workspace/submissions/VAder_1_SA_NUCES-FAST_DivideMix.txt

FOLD: 2
Selected samples: 1187
Self F1: 0.9555664117959864
Problem ID: SA-7065
Saved: /config/workspace/submissions/VAder_2_SA_NUCES-FAST_DivideMix.txt

FOLD: 3
Selected samples: 1187
Self F1: 0.9551144258854891
Problem ID: SA-7066
Saved: /config/workspace/submissions/VAder_3_SA_NUCES-FAST_DivideMix.txt

FOLD: 4
Selected samples: 1187
Self F1: 0.9596349456789607
Problem ID: SA-7068
Saved: /config/workspace/submissions/VAder_4_SA_NUCES-FAST_DivideMix.txt


# Quantum

In [4]:

import numpy as np
import uuid
import os
import time

from sklearn.datasets import load_svmlight_file
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import f1_score
from sklearn.preprocessing import normalize

import dimod
from dimod import BinaryQuadraticModel
from neal import SimulatedAnnealingSampler

from dwave.system import DWaveSampler, EmbeddingComposite
from dwave.preprocessing import ScaleComposite

from qclef import qa_access as qa

# =========================================================
# CONFIG — edit these
# =========================================================

BASE_PATH    = "/config/workspace/datasets/Tasks/2A/noise_injected_dataset/embeddings"
SUBMIT_PATH  = "/config/workspace/submissions"

USE_QPU      = True       # False = SA submission | True = QPU submission
FOLD         = 4           # 0 to 4

METHOD_ID    = "DivideMix_v1"   # same for both SA and QPU runs
GROUP_NAME   = "FAST-NUCES"          # ← change this

KEEP_RATIO     = 0.30
K              = 8
BATCH_SIZE     = 120       # ≤ 150 for QPU | up to 500 for SA
ALPHA          = 3.0
BETA           = 0.3
CONSTRAINT_STR = 8.0
SA_NUM_READS   = 500
QPU_NUM_READS  = 1
CHAIN_STRENGTH = 5.0

W_CLEAN = 0.45
W_INFO  = 0.30
W_DENS  = 0.15
W_RED   = 0.10

# =========================================================
# LOAD
# =========================================================

def load_gz(path):
    X, y = load_svmlight_file(path)
    X = X.toarray()
    y = y.astype(int)
    X = normalize(X, norm="l2")
    return X, y

# =========================================================
# SCORES
# =========================================================

def compute_score(X, y, distances, indices):
    n = len(y)

    print("  Fitting LR...")
    clf = LogisticRegression(max_iter=300, C=1.0, solver='saga', n_jobs=-1)
    clf.fit(X, y)
    proba = clf.predict_proba(X)

    loss = -np.log(proba[np.arange(n), y] + 1e-9)
    gmm  = GaussianMixture(n_components=2, random_state=42)
    gmm.fit(loss.reshape(-1, 1))
    clean_prob = gmm.predict_proba(
        loss.reshape(-1, 1))[:, np.argmin(gmm.means_)]
    clean_prob /= clean_prob.max() + 1e-9

    entropy      = -np.sum(proba * np.log(proba + 1e-12), axis=1)
    entropy_norm = entropy / (np.log(proba.shape[1]) + 1e-12)

    consistency = np.array([
        np.mean(y[indices[i, 1:]] == y[i]) for i in range(n)
    ])
    density    = 1.0 / (np.mean(distances[:, 1:], axis=1) + 1e-9)
    density   /= density.max() + 1e-9
    redundancy  = np.mean(distances[:, 1:], axis=1)
    redundancy /= redundancy.max() + 1e-9

    noise_score  = clean_prob * consistency
    noise_score /= noise_score.max() + 1e-9

    score = (W_CLEAN*noise_score + W_INFO*entropy_norm
           + W_DENS*density    - W_RED*redundancy)
    score = np.clip(score, 0, None)
    score /= score.max() + 1e-9
    return score

# =========================================================
# BATCH SPLIT
# =========================================================

def split_into_batches(n, y, batch_size):
    classes = np.unique(y)
    class_indices = {c: np.where(y == c)[0].tolist() for c in classes}
    for c in classes:
        np.random.shuffle(class_indices[c])

    interleaved = []
    while any(class_indices[c] for c in classes):
        for c in classes:
            if class_indices[c]:
                interleaved.append(class_indices[c].pop(0))

    batches = []
    for start in range(0, len(interleaved), batch_size):
        batches.append(np.array(interleaved[start:start + batch_size]))
    return batches

# =========================================================
# QUBO PER BATCH
# =========================================================

def build_batch_qubo(batch_indices, score, distances, indices, k_keep):
    global_to_local = {int(g): l for l, g in enumerate(batch_indices)}
    Q = {}

    for i in batch_indices:
        Q[(int(i), int(i))] = -ALPHA * float(score[i])

    for i in batch_indices:
        for rank in range(1, K + 1):
            j = int(indices[i, rank - 1])
            if j not in global_to_local or int(i) >= j:
                continue
            sim = max(0.0, 1.0 - float(distances[i, rank - 1]))
            key = (int(i), j)
            Q[key] = Q.get(key, 0.0) + BETA * sim

    bqm = BinaryQuadraticModel.from_qubo(Q)
    bqm.update(dimod.generators.combinations(
        list(batch_indices), k_keep, strength=CONSTRAINT_STR
    ))
    return bqm

# =========================================================
# SUBMIT VIA QCLEF  ← required for both SA and QPU
# =========================================================

def submit_sa(bqm, batch_id, fold):
    """SA submission through qclef — required alongside QPU run."""
    sampler = SimulatedAnnealingSampler()
    sampleset = qa.submit(
        sampler,
        SimulatedAnnealingSampler.sample,
        bqm,
        label=f"2A_SA_fold{fold}_b{batch_id}_{METHOD_ID}",
        num_reads=SA_NUM_READS,
    )
    best = sampleset.first.sample
    pid  = sampleset.info.get("problem_id", str(uuid.uuid4()))
    return best, pid


def submit_qpu(bqm, batch_id, fold):
    """QPU submission through qclef."""
    sampler = EmbeddingComposite(ScaleComposite(DWaveSampler()))
    sampleset = qa.submit(
        sampler,
        EmbeddingComposite.sample,
        bqm,
        label=f"2A_QPU_fold{fold}_b{batch_id}_{METHOD_ID}",
        num_reads=QPU_NUM_READS,
        chain_strength=CHAIN_STRENGTH,
    )
    best = sampleset.first.sample
    pid  = sampleset.info.get("problem_id", str(uuid.uuid4()))
    return best, pid

# =========================================================
# BATCHED SOLVE
# =========================================================

def solve_batched(score, distances, indices, y, n, fold):
    np.random.seed(42)
    batches   = split_into_batches(n, y, BATCH_SIZE)
    n_batches = len(batches)
    solver_tag = "QPU" if USE_QPU else "SA"

    print(f"\n  {n} docs → {n_batches} batches | solver={solver_tag}\n")

    all_selected = []
    all_pids     = []

    for b_idx, batch_indices in enumerate(batches):
        b_size = len(batch_indices)
        k_keep = max(1, int(KEEP_RATIO * b_size))

        print(f"  Batch {b_idx+1}/{n_batches} "
              f"(size={b_size}, selecting={k_keep})...", end=" ", flush=True)

        t0  = time.time()
        bqm = build_batch_qubo(batch_indices, score, distances, indices, k_keep)

        try:
            if USE_QPU:
                best, pid = submit_qpu(bqm, b_idx, fold)
            else:
                best, pid = submit_sa(bqm, b_idx, fold)

        except Exception as e:
            # SA fallback — still goes through qclef if possible
            print(f"\n    WARNING: submit failed ({e}), falling back to local SA")
            ss  = dimod.SimulatedAnnealingSampler().sample(
                bqm, num_reads=SA_NUM_READS)
            best = ss.first.sample
            pid  = str(uuid.uuid4())

        selected = [i for i, v in best.items() if v == 1]
        all_selected.extend(selected)
        all_pids.append(pid)

        print(f"selected={len(selected)} [{time.time()-t0:.1f}s] pid={pid[:8]}...")

    return sorted(all_selected), all_pids

# =========================================================
# WRITE SUBMISSION FILE
# =========================================================

def write_submission(selected, pids, fold, method):
    """
    Filename: VaderNYT_Fold{N}_{Method}_{Group}_{MethodID}.txt
    Contents: doc index per line, [pid1, pid2, ...] on last line
    """
    os.makedirs(SUBMIT_PATH, exist_ok=True)
    fname = f"Vader_{fold}_{method}_{GROUP_NAME}_{METHOD_ID}.txt"
    fpath = os.path.join(SUBMIT_PATH, fname)

    with open(fpath, "w") as f:
        for idx in selected:
            f.write(f"{idx}\n")
        f.write(str(pids) + "\n")

    print(f"\n  Submission written → {fpath}")
    return fpath

# =========================================================
# MAIN
# =========================================================

solver_tag = "QPU" if USE_QPU else "SA"

print(f"\n{'='*55}")
print(f"Task 2A | Fold={FOLD} | Solver={solver_tag} | Method={METHOD_ID}")
print(f"{'='*55}")

# Load
print("\n[1/4] Loading data...")
X, y = load_gz(f"{BASE_PATH}/train{FOLD}.gz")
n = len(y)
print(f"  n={n} | features={X.shape[1]} | classes={len(np.unique(y))}")

# kNN
print("\n[2/4] Building kNN graph...")
t0  = time.time()
knn = NearestNeighbors(n_neighbors=K, metric="cosine", algorithm="brute", n_jobs=-1)
knn.fit(X)
distances, indices = knn.kneighbors(X)
print(f"  Done: {time.time()-t0:.1f}s")

# Scores
print("\n[3/4] Computing quality scores...")
score = compute_score(X, y, distances, indices)

# Solve
print("\n[4/4] Solving batches via qclef...")
selected, pids = solve_batched(score, distances, indices, y, n, FOLD)

# Write submission
method_label = "QA" if USE_QPU else "SA"
fpath = write_submission(selected, pids, FOLD, method_label)

# Validate
print("\n[Validation]")
clf = LogisticRegression(max_iter=300, solver='saga', n_jobs=-1)
clf.fit(X[selected], y[selected])
f1 = f1_score(y, clf.predict(X), average="macro")
print(f"  Selected : {len(selected)} / {n} ({len(selected)/n*100:.1f}%)")
print(f"  Macro-F1 : {f1:.4f}")
print(f"\n  Run again with USE_QPU={'True' if not USE_QPU else 'False'} "
      f"to generate the {'QPU' if not USE_QPU else 'SA'} submission.")


Task 2A | Fold=4 | Solver=QPU | Method=DivideMix_v1

[1/4] Loading data...
  n=3958 | features=768 | classes=2

[2/4] Building kNN graph...
  Done: 4.5s

[3/4] Computing quality scores...
  Fitting LR...

[4/4] Solving batches via qclef...

  3958 docs → 33 batches | solver=QPU

  Batch 1/33 (size=120, selecting=36)... selected=43 [88.9s] pid=c5f18e1b...
  Batch 2/33 (size=120, selecting=36)... selected=47 [119.5s] pid=b44f4efa...
  Batch 3/33 (size=120, selecting=36)... selected=51 [101.1s] pid=444bdea8...
  Batch 4/33 (size=120, selecting=36)... selected=47 [90.0s] pid=05c6aabe...
  Batch 5/33 (size=120, selecting=36)... selected=41 [196.0s] pid=86dd1172...
  Batch 6/33 (size=120, selecting=36)... selected=56 [172.8s] pid=d5cc49e8...
  Batch 7/33 (size=120, selecting=36)... selected=45 [57.7s] pid=ed7b9bc7...
  Batch 8/33 (size=120, selecting=36)... selected=46 [144.7s] pid=4c537e06...
  Batch 9/33 (size=120, selecting=36)... selected=43 [164.3s] pid=6e96daf2...
  Batch 10/33 (size=